<a href="https://colab.research.google.com/github/siinwook/Dacon/blob/main/Track1_%EC%95%8C%EA%B3%A0%EB%A6%AC%EC%A6%98_%EB%B6%80%EB%AC%B8_K%EB%A6%AC%EA%B7%B8_%EC%84%9C%EC%9A%B8%EC%8B%9C%EB%A6%BD%EB%8C%80_%EA%B3%B5%EA%B0%9C_AI_%EA%B2%BD%EC%A7%84%EB%8C%80%ED%9A%8C_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder

import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import Dataset, DataLoader

In [ ]:
train=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Dacon/Track1 알고리즘 부문 : K리그-서울시립대 공개 AI 경진대회/train.csv')
test=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Dacon/Track1 알고리즘 부문 : K리그-서울시립대 공개 AI 경진대회/test.csv')

In [ ]:
train.head(50)

,game_id,period_id,episode_id,time_seconds,team_id,player_id,action_id,type_name,result_name,start_x,start_y,end_x,end_y,is_home,game_episode
0,126283,1,1,0.6670,2354,344559,0,Pass,Successful,52.418205,33.485444,31.322445,38.274752,True,126283_1
1,126283,1,1,3.6670,2354,250036,2,Pass,Successful,32.013240,38.100808,37.371285,30.632980,True,126283_1
2,126283,1,1,4.9680,2354,500145,4,Carry,NaN,37.371285,30.632980,38.391570,24.613144,True,126283_1
3,126283,1,1,8.2000,2354,500145,5,Pass,Successful,38.391570,24.613144,34.573350,5.545468,True,126283_1
4,126283,1,1,11.6330,2354,142106,7,Pass,Successful,34.578705,6.058256,21.274470,18.437112,True,126283_1
5,126283,1,1,13.2010,2354,500147,9,Carry,NaN,21.274470,18.437112,28.862295,24.320336,True,126283_1
6,126283,1,1,18.9000,2354,500147,10,Pass,Successful,28.862295,24.320336,26.569410,35.190204,True,126283_1
7,126283,1,1,19.9680,2354,250036,12,Carry,NaN,26.569410,35.190204,35.039130,34.624580,True,126283_1
8,126283,1,1,25.9330,2354,250036,13,Pass,Successful,35.039130,34.624580,33.001500,18.885028,True,126283_1
9,126283,1,1,29.7670,2354,500147,15,Pass,Successful,35.220780,19.540344,35.104335,33.618112,True,126283_1


In [ ]:
train['is_same'] = train['start_y'] + train['start_x'] == train['end_y'] + train['end_x']
train['is_same'].value_counts()
train.loc[train['is_same'], 'type_name'].value_counts()
#train.loc[train['type_name']=='Aerial Clearance']

,count
type_name,
Recovery,14666
Interception,9305
Tackle,7768
Duel,7664
Intervention,6038
Block,3983
Error,1647
Catch,1017
Take-On,987


In [ ]:
TRAIN_PATH = "/content/drive/MyDrive/Colab Notebooks/Dacon/Track1 알고리즘 부문 : K리그-서울시립대 공개 AI 경진대회/train.csv"
BATCH_SIZE = 64
EPOCHS = 10
LR = 1e-3
HIDDEN_DIM = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

Using device: cuda


In [ ]:
def preprocess(df):
  # Nan 처리
  df['end_x'].fillna(-1)
  df['end_y'].fillna(-1)

# 정규화
  df['start_x'] = df['start_x'] / 105.0
  df['start_y'] = df['start_y'] / 68.0
  df['end_x'] = df['end_x'] / 105.0
  df['end_y'] = df['end_y'] / 68.0
  df['time_seconds'] = df['time_seconds'] / 2700.0

# 마지막 패스 팀 레이블
  df['target_team_id'] = df.groupby('game_episode')['team_id'].transform('last') # 에피소드별 마지막 패스하는 팀 id 찾기
  df['is_lastpass_team'] = df['team_id'] == df['target_team_id']

# 마지막 패스 팀 기준 1 - x, 1 - y
  features_to_flip = ['start_x', 'start_y', 'end_x', 'end_y']
  for col in features_to_flip:
    # df.loc[행조건, 열이름] = 값
    df.loc[~df['is_lastpass_team'], col] = 1.0 - df.loc[~df['is_lastpass_team'], col]
  df['is_lastpass_team'] = df['is_lastpass_team'].map({True: 1, False: 0})

# 이동 없는 행 레이블
  df['no_movement'] = df['start_y'] + df['start_x'] == df['end_y'] + df['end_x']

# 골대/원점 기준 거리/각도 컬럼 생성
  df['s_distance_to_goal'] = np.sqrt((df['start_x'] - 1.0) ** 2 + (df['start_y'] - 0.5) ** 2)
  df['e_distance_to_goal'] = np.sqrt((df['end_x'] - 1.0) ** 2 + (df['end_y'] - 0.5) ** 2)
  df['s_angle_to_center'] = np.arctan2(df['start_y'] - 0.5, df['start_x'] - 0.5)
  df['e_angle_to_center'] = np.arctan2(df['end_y'] - 0.5, df['end_x'] - 0.5)
  df['s_distance_to_center'] = np.sqrt((df['start_x'] - 0.5) ** 2 + (df['start_y'] - 0.5) ** 2)
  df['e_distance_to_center'] = np.sqrt((df['end_x'] - 0.5) ** 2 + (df['end_y'] - 0.5) ** 2)

  return df

In [ ]:
df=train.copy()
df = preprocess(df)
df = df.sort_values(["game_episode", "time_seconds"]).reset_index(drop=True)

episodes = []
targets = []

for _, g in tqdm(df.groupby("game_episode")):
    g = g.reset_index(drop=True)
    if len(g) < 2:
        continue

    coords = []
    for i in range(len(g)):
        # 시작점 정보
        sx = g["start_x"].values[i]
        sy = g["start_y"].values[i]
        time_seconds = g["time_seconds"].values[i]
        is_lastpass_team = g["is_lastpass_team"].values[i]
        s_distance_to_goal = g["s_distance_to_goal"].values[i]
        s_angle_to_center = g["s_angle_to_center"].values[i]
        s_distance_to_center = g["s_distance_to_center"].values[i]



        coords.append([sx, sy, time_seconds, is_lastpass_team, s_distance_to_goal, s_angle_to_center, s_distance_to_center])

        # target 제외 끝점 정보
        if i < len(g) - 1:
          if g['no_movement'].values[i]: # 이동 없는 행은 좌표 한개만 넣기
            continue

          ex = g["end_x"].values[i]
          ey = g["end_y"].values[i]
          e_distance_to_goal = g["e_distance_to_goal"].values[i]
          e_angle_to_center = g["e_angle_to_center"].values[i]
          e_distance_to_center = g["e_distance_to_center"].values[i]

          coords.append([ex, ey, time_seconds, is_lastpass_team, e_distance_to_goal, e_angle_to_center, e_distance_to_center])

    seq = np.array(coords, dtype="float32")        # [T, 7]
    target_x = g["end_x"].values[-1]
    target_y = g["end_y"].values[-1]
    target = np.array([target_x, target_y], dtype="float32")  # 마지막 행 end_x, end_y

    episodes.append(seq)
    targets.append(target)

print("에피소드 수 : ", len(episodes))

100%|██████████| 15435/15435 [00:20<00:00, 762.57it/s]

에피소드 수 :  15428


In [ ]:
train[30:55]

,game_id,period_id,episode_id,time_seconds,team_id,player_id,action_id,type_name,result_name,start_x,...,is_same,target_team_id,is_lastpass_team,no_movement,s_distance_to_goal,e_distance_to_goal,s_angle_to_center,e_angle_to_center,s_distance_to_center,e_distance_to_center
30,126283,1,1,0.030333,4639,188346,44,Recovery,NaN,0.808817,...,True,2354,0,True,0.300519,0.300519,0.644020,0.644020,0.386171,0.386171
31,126283,1,1,0.030334,4639,188346,45,Carry,NaN,0.808817,...,False,2354,0,False,0.300519,0.249483,0.644020,0.482711,0.386171,0.351351
32,126283,1,1,0.030691,4639,188346,46,Pass,Successful,0.811206,...,False,2354,0,False,0.249483,0.151630,0.482711,0.123773,0.351351,0.357678
33,126283,1,1,0.031395,4639,250899,48,Pass,Successful,0.851664,...,False,2354,0,False,0.151067,0.076042,0.081141,-0.149227,0.352825,0.478430
34,126283,1,1,0.032173,4639,142258,50,Pass,Unsuccessful,0.973113,...,False,2354,0,False,0.076042,0.609273,-0.149227,-2.460458,0.478430,0.133194
35,126283,1,1,0.034642,2354,250036,51,Recovery,NaN,0.395480,...,True,2354,1,True,0.612698,0.612698,-2.379422,-2.379422,0.144497,0.144497
36,126283,1,1,0.035346,2354,250036,52,Pass,Successful,0.416212,...,False,2354,1,False,0.592271,0.606656,-2.268803,-1.794214,0.130370,0.250194
37,126283,1,1,0.036271,2354,500145,54,Pass,Successful,0.458107,...,False,2354,1,False,0.608421,0.706770,-1.721091,-1.615073,0.279792,0.477883
38,126283,1,1,0.036827,2354,142106,56,Carry,NaN,0.478848,...,False,2354,1,False,0.706770,0.669983,-1.615073,-1.619128,0.477883,0.422453
39,126283,1,1,0.037994,2354,142106,57,Pass,Successful,0.479590,...,False,2354,1,False,0.669983,0.694614,-1.619128,-2.188143,0.422453,0.272935


In [ ]:
class EpisodeDataset(Dataset):
    def __init__(self, episodes, targets):
        self.episodes = episodes
        self.targets = targets

    def __len__(self):
        return len(self.episodes)

    def __getitem__(self, idx):
        seq = torch.tensor(self.episodes[idx])   # [T, 3]
        tgt = torch.tensor(self.targets[idx])    # [3]
        length = seq.size(0)
        return seq, length, tgt

def collate_fn(batch):
    seqs, lengths, tgts = zip(*batch)
    lengths = torch.tensor(lengths, dtype=torch.long)
    padded = pad_sequence(seqs, batch_first=True)  # [B, T, 3]
    tgts = torch.stack(tgts, dim=0)                # [B, 3]
    return padded, lengths, tgts

# 에피소드 단위 train / valid split
idx_train, idx_valid = train_test_split(
    np.arange(len(episodes)), test_size=0.2, random_state=42
)

episodes_train = [episodes[i] for i in idx_train]
targets_train  = [targets[i]  for i in idx_train]
episodes_valid = [episodes[i] for i in idx_valid]
targets_valid  = [targets[i]  for i in idx_valid]

train_loader = DataLoader(
    EpisodeDataset(episodes_train, targets_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)

valid_loader = DataLoader(
    EpisodeDataset(episodes_valid, targets_valid),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)

print("train episodes:", len(episodes_train), "valid episodes:", len(episodes_valid))
train_loader

train episodes: 12342 valid episodes: 3086


In [ ]:
class LSTMBaseline(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=64):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=3,
            batch_first=True, # 입출력 텐서가 (batch, seq, feature
            bidirectional=False
        )
        self.fc = nn.Linear(hidden_dim, 2)  # (x_norm, y_norm)

    def forward(self, x, lengths):
        # x: [B, T, 3], lengths: [B] B: batch size, seq 몇 개가 한 batch인지, T: seq 길이
        packed = pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)
        h_last = h_n[-1]      # [B, H] 마지막 layer의 hidden state
        out = self.fc(h_last) # [B, 3] #에피소드별 마지막 패스의 hidden state
        return out

model = LSTMBaseline(input_dim=len(episodes[0][0]), hidden_dim=HIDDEN_DIM).to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [ ]:
best_dist = float("inf")
best_model_state = None

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    total_loss = 0.0

    for X, lengths, y in tqdm(train_loader):
        X, lengths, y = X.to(DEVICE), lengths.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        pred = model(X, lengths)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)

    train_loss = total_loss / len(train_loader.dataset)

    # --- Valid: 평균 유클리드 거리 ---
    model.eval()
    dists = []

    with torch.no_grad():
        for X, lengths, y in tqdm(valid_loader):
            X, lengths, y = X.to(DEVICE), lengths.to(DEVICE), y.to(DEVICE)
            pred = model(X, lengths)

            pred_np = pred.cpu().numpy()
            true_np = y.cpu().numpy()

            pred_x = pred_np[:, 0] * 105.0
            pred_y = pred_np[:, 1] * 68.0
            true_x = true_np[:, 0] * 105.0
            true_y = true_np[:, 1] * 68.0

            dist = np.sqrt((pred_x - true_x) ** 2 + (pred_y - true_y) ** 2)
            dists.append(dist)

    mean_dist = np.concatenate(dists).mean()  # 평균 유클리드 거리

    print(
        f"[Epoch {epoch}] "
        f"train_loss={train_loss:.4f} | "
        f"valid_mean_dist={mean_dist:.4f}"
    )

    # ----- BEST MODEL 업데이트 -----
    if mean_dist < best_dist:
        best_dist = mean_dist
        best_model_state = model.state_dict().copy()
        print(f" --> Best model updated! (dist={best_dist:.4f})")

100%|██████████| 49/49 [00:02<00:00, 20.59it/s]


[Epoch 1] train_loss=0.0495 | valid_mean_dist=17.4488
 --> Best model updated! (dist=17.4488)


100%|██████████| 49/49 [00:02<00:00, 19.22it/s]


[Epoch 2] train_loss=0.0327 | valid_mean_dist=17.6675


  9%|▉         | 18/193 [00:02<00:25,  6.98it/s]


KeyboardInterrupt: 

In [ ]:
# Best Model Load
model.load_state_dict(best_model_state)
model.eval()

test_meta = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Dacon/Track1 알고리즘 부문 : K리그-서울시립대 공개 AI 경진대회/test.csv")
submission = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Dacon/Track1 알고리즘 부문 : K리그-서울시립대 공개 AI 경진대회/sample_submission.csv")

submission = submission.merge(test_meta, on="game_episode", how="left")

preds_x, preds_y = [], []

# 1. 경로 설정
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/Dacon/Track1 알고리즘 부문 : K리그-서울시립대 공개 AI 경진대회"

# 2. 반복문 들어가기 전에 path 컬럼 일괄 수정
submission["path"] = submission["path"].apply(lambda x: x.replace("./", f"{DATA_PATH}/"))

for _, row in tqdm(submission.iterrows(), total=len(submission)):
    g = pd.read_csv(row["path"]).reset_index(drop=True)
    # 정규화된 좌표 준비

    g = preprocess(g)

    coords = []
    for i in range(len(g)):
        # 시작점 정보
        sx = g["start_x"].values[i]
        sy = g["start_y"].values[i]
        time_seconds = g["time_seconds"].values[i]
        is_lastpass_team = g["is_lastpass_team"].values[i]
        s_distance_to_goal = g["s_distance_to_goal"].values[i]
        s_angle_to_center = g["s_angle_to_center"].values[i]
        s_distance_to_center = g["s_distance_to_center"].values[i]

        coords.append([sx, sy, time_seconds, is_lastpass_team, s_distance_to_goal, s_angle_to_center, s_distance_to_center])

        # 끝점 정보
        if i < len(g) - 1:
          if g['no_movement'].values[i]: # 이동 없는 행은 좌표 한개만 넣기
            continue

          ex = g["end_x"].values[i]
          ey = g["end_y"].values[i]
          e_distance_to_goal = g["e_distance_to_goal"].values[i]
          e_angle_to_center = g["e_angle_to_center"].values[i]
          e_distance_to_center = g["e_distance_to_center"].values[i]

          coords.append([ex, ey, time_seconds, is_lastpass_team, e_distance_to_goal, e_angle_to_center, e_distance_to_center])


    seq = np.array(coords, dtype="float32")  # [T, 2]

    x = torch.tensor(seq).unsqueeze(0).to(DEVICE)      # [1, T, 2]
    length = torch.tensor([seq.shape[0]]).to(DEVICE)   # [1]

    with torch.no_grad():
        pred = model(x, length).cpu().numpy()[0] # [x_norm, y_norm]

    last_is_lastpass_team = g["is_lastpass_team"].values[-1]
    if not last_is_lastpass_team: # 뒤집혔던 팀이라면
        pred[0] = 1.0 - pred[0]
        pred[1] = 1.0 - pred[1]

    preds_x.append(pred[0] * 105.0)
    preds_y.append(pred[1] * 68.0)

print("Inference Done.")

In [ ]:
submission["end_x"] = preds_x
submission["end_y"] = preds_y
#submission[["game_episode", "end_x", "end_y"]].to_csv("/content/drive/MyDrive/Colab Notebooks/Dacon/Track1 알고리즘 부문 : K리그-서울시립대 공개 AI 경진대회/submission_new2.csv", index=False)
print("Saved: submission_LRinversion.csv")

In [ ]:
#train_loss=0.0299 | valid_mean_dist=15.9435